## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 81 samples and 13598 columns.
First few rows of the data:


,death.time..month.,death.event,relapse.time..month.,relapse.event.1,sample_title,tissue,cell line,genotype/variation,date.of.birth,gender,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,39.000000,1,17.466667,1.0,A271,lung cancer,NaN,NaN,9/28/1941,F,...,8.709060,6.720906,7.794208,8.700092,6.659364,7.210290,9.130210,7.788665,6.820204,7.948768
1,6.533333,1,4.900000,1.0,A272,lung cancer,NaN,NaN,9/11/1947,F,...,8.536846,7.101716,8.194533,9.164981,7.237020,7.072964,8.914102,7.098715,5.951072,8.348900
2,66.700000,1,58.166667,1.0,A306,lung cancer,NaN,NaN,2/20/1948,F,...,9.092578,7.227904,8.330834,9.027126,6.730124,7.091477,8.913696,6.769134,5.779236,8.045843
3,50.133333,1,50.133333,0.0,A438,lung cancer,NaN,NaN,6/16/1938,F,...,8.090881,6.188668,7.288823,8.461460,5.855103,6.579545,8.449606,6.560251,5.589890,7.269757
4,40.966667,1,33.333333,1.0,A442,lung cancer,NaN,NaN,6/16/1945,F,...,9.091247,7.010743,8.400720,9.252639,7.285403,6.846199,9.130345,7.211239,6.716373,8.030868


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
